# Four sources: load five models and average their test predictions

Standalone inference notebook requiring only **five CV checkpoints and test.pt in v4 format**.
No train.pt, raw data, or fold files are needed. Accepts `best_model.pt` from the original
CV or ensemble notebook, as well as resume checkpoints from which the validation-selected
`best_checkpoint` is extracted instead of the final epoch weights.

Set TEST_PATH and CHECKPOINT_PATHS below, then Run All. The five files must cover
folds 0..4 with the same dataset, experiment, and architecture. Bare state_dict files and models
with different architectures are rejected. Model and bank code is embedded in the library cells.
Models are loaded one at a time and share the test bank to reduce memory usage.

Output: `ensemble_predictions.csv` (ID, y_true, y_pred_fold_0..4, y_pred_mean),
`ensemble_metrics.json`/`.csv` (MSE, RMSE, CI, Pearson). **CI means concordance index**,
not a confidence interval. Undefined metrics, such as CI when all labels are equal,
are stored as null. Test labels are required for metrics; keep the original scale in test.pt.


In [ ]:
from pathlib import Path

DATASET = 'davis'  # 'davis' or 'kiba'
REPO_ROOT = next((p for base in (Path.cwd(), *Path.cwd().parents)
                  for p in (base, base / 'DATN_DTA')
                  if (p / 'training.py').is_file() and (p / 'models').is_dir()), Path.cwd())

# On Kaggle, replace these with the corresponding Add Input paths.
TEST_PATH = REPO_ROOT / 'data/four_source_embeddings' / DATASET / 'test.pt'
MODEL_DIR = REPO_ROOT / 'outputs' / f'{DATASET}_four_source_cv_ensemble'
CHECKPOINT_PATHS = [MODEL_DIR / f'fold_{i}' / 'best_model.pt' for i in range(5)]
# Alternatively, specify five Paths from different directories; file order does not matter.

OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else REPO_ROOT / 'outputs') / f'{DATASET}_ensemble_predict'
TEST_BATCH_SIZE = 512  # Reduce if GPU memory is insufficient.
CUDA_DEVICE = 'cuda:0'  # Fall back to CPU when CUDA is unavailable.
MAX_GPUS = 1

if len(CHECKPOINT_PATHS) != 5:
    raise ValueError('Exactly five checkpoints are required')
for path in [TEST_PATH, *CHECKPOINT_PATHS]:
    if not Path(path).is_file():
        raise FileNotFoundError(f'File not found: {path}; update TEST_PATH/CHECKPOINT_PATHS.')


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch-geometric'])


In [ ]:
from pathlib import Path


from types import SimpleNamespace


import csv


import hashlib


import json


import random


import numpy as np


import torch


import torch.nn as nn


class FourSourceSplit:
    """Load entities once; after GPU upload, retain indices, labels, and IDs for CSV export."""
    def __init__(self, path, expected_split, dataset=None, protein_embedding_dim=640):
        self.data = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        if self.data.get('format_version') != 4:
            raise ValueError(f'{path}: expected a v4 split file generated by create_data.py')
        self.dataset = self.data.get('dataset')
        if (self.dataset not in ('davis', 'kiba') or self.data.get('split') != expected_split
                or (dataset is not None and self.dataset != dataset)):
            raise ValueError(f'{path}: dataset/split does not match {dataset or "davis/kiba"} {expected_split}')
        self.ligand_ids = self.data['ligand_ids']
        self.protein_ids = self.data['protein_ids']
        self.pairs = self.data['pairs']
        self.labels = self.pairs['y'].float().reshape(-1)
        n = len(self.labels)
        if (self.pairs['y'].ndim != 1 or n == 0 or len(self.pairs['ligand_index']) != n
                or len(self.pairs['protein_index']) != n or len(self.pairs['row_id']) != n
                or not torch.isfinite(self.labels).all()):
            raise ValueError(f'{path}: invalid pair or label tables')
        nl, np_ = len(self.data['ligand_ids']), len(self.data['protein_ids'])
        if (len(self.data['ligand_graphs']) != nl or len(self.data['ligand_embeddings']) != nl
                or len(self.data['protein_graphs']) != np_
                or len(self.data['protein_embeddings']) != np_
                or len(self.data['protein_embedding_ids']) != np_
                or len(set(self.ligand_ids)) != nl or len(set(self.protein_ids)) != np_):
            raise ValueError(f'{path}: four-source table dimensions do not match')
        for key, size in [('ligand_index', nl), ('protein_index', np_)]:
            indices = self.pairs[key]
            if (indices.dtype != torch.long or indices.shape != (n,)
                    or indices.min() < 0 or indices.max() >= size):
                raise ValueError(f'{path}: invalid {key} indices')
        # Both sources contain unpadded token matrices; normalize dtype once per entity.
        for key, width in [('ligand_embeddings', 512),
                           ('protein_embeddings', protein_embedding_dim)]:
            for i, value in enumerate(self.data[key]):
                if (not isinstance(value, torch.Tensor) or value.ndim != 2
                        or value.shape[0] == 0 or value.shape[1] != width
                        or not value.is_floating_point() or not torch.isfinite(value).all()):
                    raise ValueError(f'{path}: {key}[{i}] must be finite with shape [L>0,{width}]')
                self.data[key][i] = value.detach().float()

    def __len__(self):
        return len(self.labels)

    def pair_ids(self, index):
        li = int(self.pairs['ligand_index'][index])
        pi = int(self.pairs['protein_index'][index])
        return self.ligand_ids[li], self.protein_ids[pi]


def pack_segments(values):
    """Concatenate segments once on CPU and store offsets without padding or pooling tokens."""
    lengths = torch.tensor([value.shape[0] for value in values], dtype=torch.long)
    ptr = torch.cat((torch.zeros(1, dtype=torch.long), lengths.cumsum(0)))
    return torch.cat(values, dim=0).contiguous(), ptr


def pack_graphs(graphs, width):
    for graph in graphs:
        x, edges = graph['x'], graph['edge_index']
        if (x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != width
                or not torch.isfinite(x).all() or edges.ndim != 2 or edges.shape[0] != 2
                or edges.dtype != torch.long
                or (edges.numel() and (edges.min() < 0 or edges.max() >= len(x)))):
            raise ValueError(f'Graph requires finite x with shape [N>0,{width}] and valid edge_index')
    x, node_ptr = pack_segments([graph['x'].detach().float() for graph in graphs])
    # Edges store LOCAL node indices; add offsets only when selecting graphs for a batch.
    edges, edge_ptr = pack_segments([graph['edge_index'].t() for graph in graphs])
    return {'x': x, 'node_ptr': node_ptr, 'edges': edges, 'edge_ptr': edge_ptr}


def same_graph(a, b):
    return torch.equal(a['x'], b['x']) and torch.equal(a['edge_index'], b['edge_index'])


def build_cpu_tables(splits):
    """Merge train/test entities; use original graph IDs and sequence-hash protein embedding IDs."""
    if len({split.dataset for split in splits}) != 1:
        raise ValueError('DAVIS and KIBA cannot share one bank')
    ligands, proteins, protein_embeddings = {}, {}, {}
    for split in splits:
        data = split.data
        for i, lid in enumerate(split.ligand_ids):
            graph, embedding = data['ligand_graphs'][i], data['ligand_embeddings'][i]
            if lid in ligands:
                old_graph, old_embedding = ligands[lid]
                if not same_graph(old_graph, graph) or not torch.equal(old_embedding, embedding):
                    raise ValueError(f'Ligand {lid}: train/test data differ')
            else:
                ligands[lid] = (graph, embedding)
        for i, pid in enumerate(split.protein_ids):
            graph = data['protein_graphs'][i]
            eid = data['protein_embedding_ids'][i]
            embedding = data['protein_embeddings'][i]
            if pid in proteins:
                old_graph, old_eid = proteins[pid]
                if old_eid != eid or not same_graph(old_graph, graph):
                    raise ValueError(f'Protein {pid}: train/test data differ')
            else:
                proteins[pid] = (graph, eid)
            if eid in protein_embeddings:
                if not torch.equal(protein_embeddings[eid], embedding):
                    raise ValueError(f'Embedding {eid}: same ID has different values')
            else:
                protein_embeddings[eid] = embedding
    ligand_ids, protein_ids, embedding_ids = sorted(ligands), sorted(proteins), sorted(protein_embeddings)
    ligand_map = {key: i for i, key in enumerate(ligand_ids)}
    protein_map = {key: i for i, key in enumerate(protein_ids)}
    embedding_map = {key: i for i, key in enumerate(embedding_ids)}
    tables = {}
    for prefix, graphs, width in (
            ('ligand', [ligands[key][0] for key in ligand_ids], 78),
            ('protein', [proteins[key][0] for key in protein_ids], 41)):
        for key, value in pack_graphs(graphs, width).items():
            tables[f'{prefix}_{key}'] = value
    tables['ligand_embedding'], tables['ligand_embedding_ptr'] = pack_segments(
        [ligands[key][1] for key in ligand_ids])
    tables['protein_embedding'], tables['protein_embedding_ptr'] = pack_segments(
        [protein_embeddings[key] for key in embedding_ids])
    tables['protein_to_embedding'] = torch.tensor(
        [embedding_map[proteins[key][1]] for key in protein_ids], dtype=torch.long)
    return tables, ligand_map, protein_map


def execution_devices(device, use_data_parallel=True, max_gpus=2):
    """Put the primary GPU first; default to at most two GPUs, with single-GPU/CPU fallback."""
    device = torch.device(device)
    if max_gpus < 1:
        raise ValueError('MAX_GPUS must be >= 1')
    if device.type != 'cuda':
        return [device]
    count = torch.cuda.device_count()
    primary = device.index if device.index is not None else torch.cuda.current_device()
    if not 0 <= primary < count:
        raise ValueError(f'Primary GPU cuda:{primary} is not among the {count} available GPUs')
    ids = [primary]
    if use_data_parallel:
        ids += [i for i in range(count) if i != primary]
    return [torch.device('cuda', i) for i in ids[:max_gpus]]


def take_segments(values, ptr, indices):
    """Look up segments and build batch vectors on the bank's device (GPU during training)."""
    starts = ptr.index_select(0, indices)
    lengths = ptr.index_select(0, indices + 1) - starts
    batch_ptr = torch.cat((lengths.new_zeros(1), lengths.cumsum(0)))
    owners = torch.repeat_interleave(torch.arange(indices.numel(), device=indices.device), lengths)
    local = torch.arange(owners.numel(), device=indices.device) - batch_ptr.index_select(0, owners)
    positions = starts.index_select(0, owners) + local
    return values.index_select(0, positions), owners, batch_ptr


class DeviceFourSourceStore:
    """Each GPU holds a tensor dictionary; replicas share a read-only Python store."""
    def __init__(self, cpu_tables, devices, ligand_map, protein_map):
        self.devices = tuple(torch.device(device) for device in devices)
        if not self.devices or len(set(self.devices)) != len(self.devices):
            raise ValueError('Expected a non-empty list of distinct devices')
        self.device = self.devices[0]  # Pair indices and labels reside on the primary GPU.
        self.ligand_id_to_index = ligand_map
        self.protein_id_to_index = protein_map
        self.nbytes = sum(value.numel() * value.element_size() for value in cpu_tables.values())
        self.tables = {}
        for target in self.devices:
            if target.type == 'cuda':
                free, _ = torch.cuda.mem_get_info(target)
                if self.nbytes > free:
                    raise RuntimeError(f'Bank on {target} requires {self.nbytes / 1024**3:.3f} GiB, '
                                       f'but only {free / 1024**3:.3f} GiB is free. Restart the session to release the old bank.')
            # Upload once per GPU. The bank is not a model buffer, so DataParallel
            # does not broadcast/copy it on each forward pass, and checkpoints exclude it.
            self.tables[str(target)] = {
                key: value.detach().to(target) for key, value in cpu_tables.items()}
            print(f'Four sources on {target}: {self.nbytes / 1024**3:.3f} GiB; '
                  f'{len(ligand_map)} ligand graphs, {len(protein_map)} protein graphs, '
                  f'{cpu_tables["protein_embedding_ptr"].numel() - 1} protein embeddings')

    def on(self, device):
        key = str(torch.device(device))
        if key not in self.tables:
            raise ValueError(f'Bank is not prepared on {key}')
        return self.tables[key]

    def graph_batch(self, prefix, indices):
        t = self.on(indices.device)
        x, batch, node_ptr = take_segments(t[f'{prefix}_x'], t[f'{prefix}_node_ptr'], indices)
        edges, edge_batch, _ = take_segments(t[f'{prefix}_edges'], t[f'{prefix}_edge_ptr'], indices)
        edge_index = (edges + node_ptr.index_select(0, edge_batch).unsqueeze(1)).t().contiguous()
        return x, edge_index, batch

    def lookup(self, pair_indices):
        # DataParallel splits INDICES; each replica looks up its local batch on the same device.
        t = self.on(pair_indices.device)
        li, pi = pair_indices[:, 0], pair_indices[:, 1]
        drug_x, drug_edges, drug_batch = self.graph_batch('ligand', li)
        prot_x, prot_edges, prot_batch = self.graph_batch('protein', pi)
        drug_embedding, drug_embedding_batch, _ = take_segments(
            t['ligand_embedding'], t['ligand_embedding_ptr'], li)
        embedding_indices = t['protein_to_embedding'].index_select(0, pi)
        protein_embedding, protein_embedding_batch, _ = take_segments(
            t['protein_embedding'], t['protein_embedding_ptr'], embedding_indices)
        # Tensors already reside on the GPU; wrap attributes without CPU collation or copying.
        return (SimpleNamespace(x=drug_x, edge_index=drug_edges, batch=drug_batch,
                     drug_embedding=drug_embedding, drug_embedding_batch=drug_embedding_batch),
                SimpleNamespace(x=prot_x, edge_index=prot_edges, batch=prot_batch,
                     protein_embedding=protein_embedding, protein_embedding_batch=protein_embedding_batch))


class IndexBatches:
    """Yield only GPU pair indices and labels per batch; do not fetch graphs/tokens on CPU."""
    def __init__(self, split, store, batch_size, shuffle=False, seed=42):
        if batch_size < 1:
            raise ValueError('Batch size must be >= 1')
        self.size, self.batch_size = len(split), batch_size
        self.shuffle, self.device = shuffle, store.device
        self.generator = torch.Generator(device=self.device).manual_seed(seed)
        ligand_remap = torch.tensor([store.ligand_id_to_index[key] for key in split.ligand_ids])
        protein_remap = torch.tensor([store.protein_id_to_index[key] for key in split.protein_ids])
        self.pairs = torch.stack((
            ligand_remap.index_select(0, split.pairs['ligand_index']),
            protein_remap.index_select(0, split.pairs['protein_index'])), dim=1).to(self.device)
        self.labels = split.labels.reshape(-1, 1).to(self.device)

    def __len__(self):
        return (self.size + self.batch_size - 1) // self.batch_size

    def reset_seed(self, seed):
        self.generator.manual_seed(seed)

    def __iter__(self):
        order = (torch.randperm(self.size, generator=self.generator, device=self.device)
                 if self.shuffle else torch.arange(self.size, device=self.device))
        for start in range(0, self.size, self.batch_size):
            rows = order[start:start + self.batch_size]
            yield self.pairs.index_select(0, rows), self.labels.index_select(0, rows)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def dataset_signature(splits):
    # Validate pair/label order and IDs without rereading large embedding tensors.
    digest = hashlib.sha256()
    for split in splits:
        digest.update(json.dumps([split.ligand_ids, split.protein_ids],
                                 ensure_ascii=False).encode('utf-8'))
        for key in ('ligand_index', 'protein_index', 'y'):
            value = split.pairs[key].detach().cpu().contiguous()
            digest.update(str((key, tuple(value.shape), str(value.dtype))).encode())
            digest.update(value.numpy().tobytes())
    return digest.hexdigest()




In [ ]:
"""Four-source DTA model; matches the current original-copy notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.nn import global_mean_pool as gap
from torch_geometric.nn import global_max_pool as gmp
# GAT  model
class GATnGCN_Model(torch.nn.Module):
    def __init__(self, data_store, num_features_xd=78, n_output=1, num_features_xt=41,
                      output_dim=128, dropout=0.2, protein_embedding_dim=640):
        super(GATnGCN_Model, self).__init__()
        # The store is a plain Python object; state_dict contains only model weights.
        self.data_store = data_store
        if output_dim != 128:
            raise ValueError('Fusion requires output_dim=128')
        # Ligand graph: GAT 8x64 = 512 -> GAT 4x64 = 256; max+mean pooling = 512.
        self.conv1 = GATConv(num_features_xd, 64, heads=8, concat=True)
        self.conv2 = GATConv(512, 64, heads=4, concat=True)
        self.fc_g1 = torch.nn.Linear(512, 256)
        self.fc_g2 = torch.nn.Linear(256, output_dim)

        # Replace ECFP with Chemformer ligand embeddings. Pool only in forward.
        self.fc_drug_embedding = nn.Linear(2 * 512, 128)
        self.bn_drug_embedding = nn.BatchNorm1d(128)
        # Protein graph: GAT 8x64 -> GCN 256, with 41 input features and separate weights.
        self.conv_prot1 = GATConv(num_features_xt, 64, heads=8, concat=True)
        self.conv_prot2 = GCNConv(512, 256)
        self.fc_prot_g1 = nn.Linear(512, 256)
        self.fc_prot_g2 = nn.Linear(256, output_dim)

        # Protein ESM-2: concatenate max/mean pooling, then project 1280 -> 128.
        self.fc_protein_embedding = nn.Linear(2 * protein_embedding_dim, 128)
        self.bn_protein_embedding = nn.BatchNorm1d(128)

        #concat
        self.fc_xd = nn.Linear(256,128)
        self.fc_xp = nn.Linear(256,128)

        # Small prediction head: 256 -> 512 -> 128 -> affinity.
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 128)
        self.out = nn.Linear(128, n_output)

        # activation and regularization
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, pair_indices):
        # Look up all four sources on the GPU at the start of each forward pass.
        data_drug, data_protein = self.data_store.lookup(pair_indices)
        # graph input feed-forward
        x_drug, edge_index_drug, batch_drug = data_drug.x, data_drug.edge_index, data_drug.batch

        x_drug = self.conv1(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        x_drug = self.conv2(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        # apply global max pooling (gmp) and global mean pooling (gap)
        x_drug = torch.cat([gmp(x_drug, batch_drug), gap(x_drug, batch_drug)], dim=1)
        x_drug = self.relu(self.fc_g1(x_drug))
        x_drug = self.dropout(x_drug)
        x_drug = self.fc_g2(x_drug)

        # Tokens are concatenated; drug_embedding_batch identifies their samples.
        # The bank preserves raw tokens; max+mean pooling happens only here.
        # Concatenate [max, mean], pooling each sample separately: [B,1024].
        x_drug_embedding = torch.cat([
            gmp(data_drug.drug_embedding, data_drug.drug_embedding_batch),
            gap(data_drug.drug_embedding, data_drug.drug_embedding_batch),
        ], dim=1)
        x_drug_embedding = self.fc_drug_embedding(x_drug_embedding)
        # BatchNorm needs >=2 training samples; use running statistics for a singleton batch.
        if self.training and x_drug_embedding.size(0) == 1:
            x_drug_embedding = F.batch_norm(
                x_drug_embedding, self.bn_drug_embedding.running_mean,
                self.bn_drug_embedding.running_var, self.bn_drug_embedding.weight,
                self.bn_drug_embedding.bias, training=False, eps=self.bn_drug_embedding.eps)
        else:
            x_drug_embedding = self.bn_drug_embedding(x_drug_embedding)
        x_drug_embedding = self.relu(x_drug_embedding)
        x_drug_embedding = self.dropout(x_drug_embedding)

        # protein input feed-forward:
        x_prots, edge_index_prots, prots_batch= data_protein.x, data_protein.edge_index, data_protein.batch
        # Protein: GAT -> GCN -> max+mean -> Linear/ReLU/Dropout -> Linear.
        x_prots = self.conv_prot1(x_prots, edge_index_prots)
        x_prots = self.relu(x_prots)
        x_prots = self.conv_prot2(x_prots, edge_index_prots)
        x_prots = self.relu(x_prots)
        x_prots = torch.cat([gmp(x_prots, prots_batch), gap(x_prots, prots_batch)], dim=1)
        x_prots = self.relu(self.fc_prot_g1(x_prots))
        x_prots = self.dropout(x_prots)
        x_prots = self.fc_prot_g2(x_prots)
        # Pool tokens per sample using protein_embedding_batch, not the graph node batch.
        # Both pooling operations use each sample's actual tokens without padding.
        # Concatenation order: [max (640), mean (640)] -> [B,1280].
        x_protein_embedding = torch.cat([
            gmp(data_protein.protein_embedding, data_protein.protein_embedding_batch),
            gap(data_protein.protein_embedding, data_protein.protein_embedding_batch),
        ], dim=1)
        x_protein_embedding = self.fc_protein_embedding(x_protein_embedding)
        if self.training and x_protein_embedding.size(0) == 1:
            x_protein_embedding = F.batch_norm(
                x_protein_embedding, self.bn_protein_embedding.running_mean,
                self.bn_protein_embedding.running_var, self.bn_protein_embedding.weight,
                self.bn_protein_embedding.bias, training=False, eps=self.bn_protein_embedding.eps)
        else:
            x_protein_embedding = self.bn_protein_embedding(x_protein_embedding)
        x_protein_embedding = self.relu(x_protein_embedding)
        x_protein_embedding = self.dropout(x_protein_embedding)

        # Ablation: concatenate graph features directly, without cross-attention or residuals.
        xd = torch.cat((x_drug, x_drug_embedding), dim=1)       # [B,256]
        xp = torch.cat((x_prots, x_protein_embedding), dim=1)  # [B,256]
        xd = self.relu(self.fc_xd(xd))
        xp = self.relu(self.fc_xp(xp))

        xc = torch.cat((xd,xp), 1)
        # Apply ReLU/Dropout after each hidden layer; the output layer is linear.
        xc = self.fc1(xc)
        xc = self.relu(xc)
        xc = self.dropout(xc)
        xc = self.fc2(xc)
        xc = self.relu(xc)
        xc = self.dropout(xc)
        out = self.out(xc)
        return out

In [ ]:
from __future__ import annotations


import argparse


import csv


from dataclasses import dataclass, field


from pathlib import Path


import time


import numpy as np


import torch


import torch.nn as nn


import torch.nn.functional as F


MODEL_TYPE = 'GATnGCN_DrugGAT8x64_GAT4x64_ProteinGATGCN_DualEmbeddingMaxMean_SmallMLP'


def configure_model(model_config, store):

    for target in store.devices:
        store.on(target)
    core = GATnGCN_Model(data_store=store, **model_config).to(store.device)
    if len(store.devices) > 1:
        gpu_ids = [target.index for target in store.devices]
        return nn.DataParallel(core, device_ids=gpu_ids, output_device=gpu_ids[0], dim=0)
    return core


@torch.no_grad()
def predicting(model, batches):
    model.eval()
    labels, predictions = [], []
    for pair_indices, batch_labels in batches:
        output = model(pair_indices)
        predictions.append(output.cpu().reshape(-1))
        labels.append(batch_labels.cpu().reshape(-1))
    return torch.cat(labels).numpy(), torch.cat(predictions).numpy()




In [ ]:
from __future__ import annotations


import csv


from dataclasses import dataclass


import gc


import hashlib


import json


from pathlib import Path


import time


import numpy as np


import torch


def regression_metrics(labels, predictions):
    """MSE/RMSE/Pearson and exact CI in O(n log n), excluding equal-label pairs."""
    y, p = np.asarray(labels, dtype=np.float64), np.asarray(predictions, dtype=np.float64)
    if (y.ndim != 1 or p.shape != y.shape or not len(y)
            or not np.isfinite(y).all() or not np.isfinite(p).all()):
        raise ValueError('Metrics require matching, non-empty, finite prediction/label vectors')
    mse = float(np.mean((y - p) ** 2))
    ranks = np.searchsorted(np.unique(p), p) + 1
    tree = np.zeros(int(ranks.max()) + 1, dtype=np.int64)

    def count(rank):
        total = 0
        while rank:
            total += int(tree[rank])
            rank -= rank & -rank
        return total

    order = np.argsort(y, kind='stable')
    concordant, comparable, previous, start = 0.0, 0, 0, 0
    while start < len(order):
        end = start + 1
        while end < len(order) and y[order[end]] == y[order[start]]:
            end += 1
        for index in order[start:end]:
            rank = int(ranks[index])
            below = count(rank - 1)
            concordant += below + 0.5 * (count(rank) - below)
            comparable += previous
        for index in order[start:end]:
            rank = int(ranks[index])
            while rank < len(tree):
                tree[rank] += 1
                rank += rank & -rank
        previous += end - start
        start = end
    pearson = float(np.corrcoef(y, p)[0, 1]) if np.ptp(y) > 0 and np.ptp(p) > 0 else None
    return {'mse': mse, 'rmse': mse ** 0.5,
            'ci': concordant / comparable if comparable else None, 'pearson': pearson}


def write_json(path, value):
    temporary = Path(path).with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
    temporary.replace(path)



import csv


import gc


import json


from pathlib import Path


import numpy as np


import torch


AGGREGATION = 'arithmetic_mean_of_5_fold_predictions_then_metrics_v1'


def ensemble_metrics(labels, predictions_by_fold):
    if set(predictions_by_fold) != set(range(5)):
        raise ValueError('Ensemble requires exactly 5 folds with distinct IDs: 0, 1, 2, 3, 4')
    labels = np.asarray(labels, dtype=np.float64)
    predictions = [np.asarray(predictions_by_fold[i], dtype=np.float64) for i in range(5)]
    # Validate every model before averaging: never silently drop NaN/missing folds.
    individual = {str(i): regression_metrics(labels, p) for i, p in enumerate(predictions)}
    matrix = np.stack(predictions, axis=1)
    mean_prediction = matrix.mean(axis=1)
    return matrix, mean_prediction, regression_metrics(labels, mean_prediction), individual


def export_ensemble(output_dir, rows, predictions_by_fold, metadata):
    """Rows and every prediction vector must refer to the same ordered test samples."""
    labels = np.asarray([float(row['affinity']) for row in rows], dtype=np.float64)
    matrix, mean_prediction, metrics, individual = ensemble_metrics(labels, predictions_by_fold)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / 'ensemble_predictions.csv'
    temporary = prediction_path.with_suffix('.csv.tmp')
    with temporary.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['row_id', 'ligand_id', 'protein_id', 'y_true',
                         *[f'y_pred_fold_{i}' for i in range(5)], 'y_pred_mean'])
        for index, row in enumerate(rows):
            writer.writerow([row['row_id'], row['ligand_id'], row['protein_id'],
                             labels[index], *matrix[index], mean_prediction[index]])
    temporary.replace(prediction_path)
    summary = {**metadata, 'aggregation': AGGREGATION, 'complete': True,
               'completed_folds': list(range(5)), 'n_test': len(labels),
               'test_metrics': metrics, 'per_model_test_metrics': individual,
               'prediction_file': prediction_path.name}
    write_json(output_dir / 'ensemble_metrics.json', summary)
    with (output_dir / 'ensemble_metrics.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['model', *metrics])
        writer.writeheader()
        for fold, values in individual.items():
            writer.writerow({'model': f'fold_{fold}', **values})
        writer.writerow({'model': 'ensemble_mean_prediction', **metrics})
    return summary


def load_fold_checkpoint(path):
    """Accept best_model.pt or extract the validation-selected best from a resume file."""
    payload = torch.load(Path(path), map_location='cpu', weights_only=True, mmap=True)
    if isinstance(payload, dict) and payload.get('checkpoint_kind') == 'training_resume_v1':
        payload = payload.get('best_checkpoint')
    required = {'fold', 'dataset', 'experiment_id', 'model_config', 'model_state_dict',
                'selection_split', 'validation_mse', 'epoch', 'model_type'}
    if not isinstance(payload, dict) or not required.issubset(payload):
        raise ValueError(f'{path}: expected a CV checkpoint with metadata (best_model.pt or resume), '
                         'not a bare state_dict or single-run checkpoint')
    if type(payload['fold']) is not int or payload['fold'] not in range(5):
        raise ValueError(f'{path}: fold must be in 0..4')
    if (payload['selection_split'] != 'validation'
            or not np.isfinite(payload['validation_mse']) or payload['epoch'] < 1):
        raise ValueError(f'{path}: invalid validation-selected checkpoint')
    if payload['model_type'] != MODEL_TYPE:
        raise ValueError(f'{path}: model architecture is not supported by this notebook')
    return payload


def validate_checkpoint_set(checkpoint_paths, dataset):
    paths = [Path(path).expanduser().resolve() for path in checkpoint_paths]
    if len(paths) != 5 or len(set(paths)) != 5:
        raise ValueError('CHECKPOINT_PATHS must contain exactly 5 files with distinct paths')
    by_fold, common = {}, None
    for path in paths:
        payload = load_fold_checkpoint(path)
        fold = payload['fold']
        if payload['dataset'] != dataset:
            raise ValueError(f'{path}: checkpoint dataset differs from {dataset}')
        if fold in by_fold:
            raise ValueError(f'Two checkpoints have the same fold {fold}; all folds 0..4 are required')
        identity = {key: payload.get(key) for key in
                    ('experiment_id', 'model_config', 'model_type', 'model_signature', 'protocol')}
        if not identity['experiment_id'] or not isinstance(identity['model_config'], dict):
            raise ValueError(f'{path}: missing experiment identity or model config')
        if common is not None and identity != common:
            raise ValueError('All 5 checkpoints must have the same experiment, architecture, and model config')
        common = identity
        by_fold[fold] = {'path': str(path), 'fold': fold, 'best_epoch': payload['epoch'],
                         'validation_mse': payload['validation_mse']}
        del payload
    return by_fold, common


def run_ensemble_inference(checkpoint_paths, test_path, output_dir, dataset='davis',
                           batch_size=512, device='cuda:0', max_gpus=1):
    """Test-only bank, one model at a time, same test row order, no training or fold JSON."""
    if dataset not in ('davis', 'kiba'):
        raise ValueError('DATASET must be davis or kiba')
    by_fold, common = validate_checkpoint_set(checkpoint_paths, dataset)
    config = common['model_config']
    test_data = FourSourceSplit(Path(test_path), 'test', dataset,
                               protein_embedding_dim=config.get('protein_embedding_dim', 640))
    signature = dataset_signature((test_data,))
    rows = [{'row_id': str(test_data.pairs['row_id'][i]),
             'ligand_id': test_data.pair_ids(i)[0], 'protein_id': test_data.pair_ids(i)[1],
             'affinity': float(test_data.labels[i])} for i in range(len(test_data))]
    tables, ligand_map, protein_map = build_cpu_tables((test_data,))
    device = torch.device(device)
    if device.type == 'cuda' and not torch.cuda.is_available():
        device = torch.device('cpu')
    devices = execution_devices(device, use_data_parallel=max_gpus > 1, max_gpus=max_gpus)
    store = DeviceFourSourceStore(tables, devices, ligand_map, protein_map)
    test_data.data = None
    del tables
    batches = IndexBatches(test_data, store, batch_size, shuffle=False)
    expected_labels = test_data.labels.numpy()
    predictions = {}
    for fold in range(5):
        item = by_fold[fold]
        payload = load_fold_checkpoint(item['path'])
        model = configure_model(config, store)
        unwrap_model(model).load_state_dict(payload['model_state_dict'], strict=True)
        del payload
        labels, predictions[fold] = predicting(model, batches)
        if not np.array_equal(labels, expected_labels):
            raise ValueError(f'Fold {fold}: test order or labels changed during inference')
        print(f'Fold {fold}, best epoch {item["best_epoch"]}: '
              f'{regression_metrics(labels, predictions[fold])}')
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    return export_ensemble(output_dir, rows, predictions, {
        'dataset': dataset, 'experiment_id': common['experiment_id'],
        'model_config': config, 'test_path': str(Path(test_path).resolve()),
        'test_signature': signature, 'checkpoints': [by_fold[i] for i in range(5)],
    })




In [ ]:
result = run_ensemble_inference(
    checkpoint_paths=CHECKPOINT_PATHS, test_path=TEST_PATH, output_dir=OUTPUT_DIR,
    dataset=DATASET, batch_size=TEST_BATCH_SIZE, device=CUDA_DEVICE, max_gpus=MAX_GPUS)

print('Metrics from the mean prediction of five models:')
for name, value in result['test_metrics'].items():
    print(f'{name}: {value}')
print('Prediction:', OUTPUT_DIR / 'ensemble_predictions.csv')
print('Metric:', OUTPUT_DIR / 'ensemble_metrics.json')

import pandas as pd
display(pd.read_csv(OUTPUT_DIR / 'ensemble_predictions.csv').head(10))
